# EDA - Credit Score

Business Problem:

> "Will this customer default on their credit line in the next 2 years?"

## Data Dictionary

| Column Name | Description | Type |
| :--- | :--- | :--- |
| _c0 | An anonymous ID/Index for the row. | Integer |
| serious_dlqin2yrs | Person experienced 90 days past due delinquency or worse (Target Variable: 1=Yes, 0=No). | Binary |
| revolving_utilization_of_unsecured_lines | Total balance on credit cards and personal lines of credit (excluding real estate) divided by the sum of credit limits. | Float |
| age | Age of borrower in years. | Integer |
| number_of_time30_59_days_past_due_not_worse | Number of times borrower has been 30-59 days past due but no worse in the last 2 years. | Integer |
| debt_ratio | Monthly debt payments, alimony, living costs divided by monthly gross income. | Float |
| monthly_income | Monthly income of the borrower. | Float |
| number_of_open_credit_lines_and_loans | Number of open loans (installment like car loan or mortgage) and lines of credit (e.g. credit cards). | Integer |
| number_of_times90_days_late | Number of times borrower has been 90 days or more past due. | Integer |
| number_real_estate_loans_or_lines | Number of mortgage and real estate loans including home equity lines of credit. | Integer |
| number_of_time60_89_days_past_due_not_worse | Number of times borrower has been 60-89 days past due but no worse in the last 2 years. | Integer |
| number_of_dependents | Number of dependents in family excluding themselves (spouse, children etc.). | Integer |


| Feature                                     | Meaning            |
| ------------------------------------------- | ------------------ |
| revolving_utilization_of_unsecured_lines    | credit utilization |
| number_of_time30_59_days_past_due_not_worse | late payments      |
| debt_ratio                                  | debt / income      |
| monthly_income                              | customer income    |
| age                                         | borrower age       |


Install packages

In [47]:
!uv pip install -q \
        pandas \
        pyarrow \
        boto3 \
        s3fs

Import packages

In [48]:
import pandas as pd

Inspect dataset

In [49]:
s3_path = "s3://data-lake/silver/credit_risk/cleaned/"
endpoint = "http://localstack:4566"

df = pd.read_parquet(
    s3_path,
    storage_options={
        "key": "test",
        "secret": "test",
        "client_kwargs": {
            "endpoint_url": endpoint
        }
    }
)

df.head()

,_c0,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
0,1228,0,0.072018143,50,0,1.434782609,2000,12,0,1,0,1
1,1611,0,0.023224944,64,0,0.391854139,6800,18,0,4,0,3
2,4580,0,0,59,0,0.001273705,7065,5,0,0,0,0
3,7436,0,0.954197875,47,1,0.300676293,5470,7,0,1,0,2
4,8885,0,0.024449389,30,0,0.002223756,13040,2,0,0,0,2


In [50]:
df.shape

(150000, 12)

In [51]:
df.columns

Index(['_c0', 'serious_dlqin2yrs', 'revolving_utilization_of_unsecured_lines',
       'age', 'number_of_time30_59_days_past_due_not_worse', 'debt_ratio',
       'monthly_income', 'number_of_open_credit_lines_and_loans',
       'number_of_times90_days_late', 'number_real_estate_loans_or_lines',
       'number_of_time60_89_days_past_due_not_worse', 'number_of_dependents'],
      dtype='str')

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                       Non-Null Count   Dtype
---  ------                                       --------------   -----
 0   _c0                                          150000 non-null  str  
 1   serious_dlqin2yrs                            150000 non-null  str  
 2   revolving_utilization_of_unsecured_lines     150000 non-null  str  
 3   age                                          150000 non-null  str  
 4   number_of_time30_59_days_past_due_not_worse  150000 non-null  str  
 5   debt_ratio                                   150000 non-null  str  
 6   monthly_income                               150000 non-null  str  
 7   number_of_open_credit_lines_and_loans        150000 non-null  str  
 8   number_of_times90_days_late                  150000 non-null  str  
 9   number_real_estate_loans_or_lines            150000 non-null  str  
 10  number_of_time60_89

In [53]:
df.describe()

,_c0,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
count,150000,150000,150000,150000,150000,150000,150000,150000,150000,150000,150000,150000
unique,150000,2,125728,86,16,114194,13595,58,19,28,13,14
top,1228,0,0,49,0,0,NA,6,0,0,0,0
freq,1,139974,10878,3837,126018,4113,29731,13614,141662,56188,142396,86902


## Target Variable

In [54]:
df["serious_dlqin2yrs"].value_counts(normalize=True)

serious_dlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

## Data Cleaning

Should be done at ingestion step at the bronze layer.

In [55]:
df = df.replace("NA", pd.NA)

In [56]:
df = df.drop(columns=["_c0"])

In [57]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                                       Non-Null Count   Dtype  
---  ------                                       --------------   -----  
 0   serious_dlqin2yrs                            150000 non-null  int64  
 1   revolving_utilization_of_unsecured_lines     150000 non-null  float64
 2   age                                          150000 non-null  int64  
 3   number_of_time30_59_days_past_due_not_worse  150000 non-null  int64  
 4   debt_ratio                                   150000 non-null  float64
 5   monthly_income                               120269 non-null  float64
 6   number_of_open_credit_lines_and_loans        150000 non-null  int64  
 7   number_of_times90_days_late                  150000 non-null  int64  
 8   number_real_estate_loans_or_lines            150000 non-null  int64  
 9   number_of_time60_89_days_past_due_not_worse  150000 non-null  int64  


## Detect Missings

**Avoid Data Leakage**

Data leakage occurs when information from outside the training dataset is used to create the model. If you calculate the mean or median of your entire dataset before splitting, the training set will "know" something about the distribution of the test set.

In [59]:
df.isnull().sum().sort_values(ascending=False)

monthly_income                                 29731
number_of_dependents                            3924
serious_dlqin2yrs                                  0
age                                                0
revolving_utilization_of_unsecured_lines           0
debt_ratio                                         0
number_of_time30_59_days_past_due_not_worse        0
number_of_open_credit_lines_and_loans              0
number_of_times90_days_late                        0
number_real_estate_loans_or_lines                  0
number_of_time60_89_days_past_due_not_worse        0
dtype: int64

In [60]:
(df.isnull().sum() / len(df)).sort_values(ascending=False)

monthly_income                                 0.198207
number_of_dependents                           0.026160
serious_dlqin2yrs                              0.000000
age                                            0.000000
revolving_utilization_of_unsecured_lines       0.000000
debt_ratio                                     0.000000
number_of_time30_59_days_past_due_not_worse    0.000000
number_of_open_credit_lines_and_loans          0.000000
number_of_times90_days_late                    0.000000
number_real_estate_loans_or_lines              0.000000
number_of_time60_89_days_past_due_not_worse    0.000000
dtype: float64

## Detect Outliers

In [61]:
df.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
90%,0.000000,0.981278,72.000000,1.000000,1267.000000,1.166600e+04,15.000000,0.000000,2.000000,0.000000,2.000000
95%,1.000000,1.000000,78.000000,2.000000,2449.000000,1.458760e+04,18.000000,1.000000,3.000000,1.000000,3.000000
99%,1.000000,1.092956,87.000000,4.000000,4979.040000,2.500000e+04,24.000000,3.000000,4.000000,2.000000,4.000000
max,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [62]:
df["age"].describe()

count    150000.000000
mean         52.295207
std          14.771866
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64

A bank cannot legally give credit to minors.

In [63]:
df[df["age"] < 18]

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
55001,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


0.5 = using 50% of credit

In [64]:
df["revolving_utilization_of_unsecured_lines"].describe()

count    150000.000000
mean          6.048438
std         249.755371
min           0.000000
25%           0.029867
50%           0.154181
75%           0.559046
max       50708.000000
Name: revolving_utilization_of_unsecured_lines, dtype: float64

In [65]:
df[df["revolving_utilization_of_unsecured_lines"] > 5]

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
1896,0,275.0,41,0,513.000000,NaN,3,0,1,0,0.0
4181,0,10151.0,35,0,0.341234,4260.0,10,0,0,0,0.0
4227,0,5649.0,63,0,0.287246,17625.0,6,0,1,0,1.0
4971,0,1761.0,64,0,0.082235,7587.0,2,0,1,0,0.0
5775,1,8328.0,39,0,21395.000000,NaN,9,0,2,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...
147234,0,3853.0,54,0,0.119989,3791.0,4,0,0,0,2.0
148083,0,815.0,63,0,1365.000000,NaN,3,0,1,0,0.0
148092,0,1620.0,40,0,0.206040,9900.0,6,0,1,0,2.0
149095,0,5186.0,43,0,0.211158,5000.0,4,0,1,0,2.0


In [66]:
df["debt_ratio"].describe()

count    150000.000000
mean        353.005076
std        2037.818523
min           0.000000
25%           0.175074
50%           0.366508
75%           0.868254
max      329664.000000
Name: debt_ratio, dtype: float64

- debt_ratio = debt / income
- Values like 1000+ are almost always data errors.

In [67]:
df[df["debt_ratio"] > 100]

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
17,0,0.023967,37,0,408.0,NaN,6,0,0,0,0.0
20,0,0.001476,49,0,2940.0,NaN,5,0,2,0,0.0
23,0,0.019527,66,0,6049.0,NaN,12,0,4,0,0.0
27,0,0.146401,66,0,2735.0,NaN,16,0,1,0,0.0
31,0,0.003550,64,0,2471.0,NaN,11,0,2,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
149973,0,0.486457,46,0,192.0,NaN,4,0,0,0,1.0
149979,0,0.021966,62,0,5762.0,NaN,15,0,4,0,NaN
149980,0,0.191232,69,0,1727.0,NaN,10,0,2,0,0.0
149986,0,0.036018,54,0,2274.0,NaN,12,0,1,0,0.0


In [68]:
df[
[
"number_of_time30_59_days_past_due_not_worse",
"number_of_time60_89_days_past_due_not_worse",
"number_of_times90_days_late"
]
].describe()

,number_of_time30_59_days_past_due_not_worse,number_of_time60_89_days_past_due_not_worse,number_of_times90_days_late
count,150000.000000,150000.000000,150000.000000
mean,0.421033,0.240387,0.265973
std,4.192781,4.155179,4.169304
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,98.000000,98.000000,98.000000


In [69]:
df[df["number_of_times90_days_late"] > 20]

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
70,0,1.0,22,98,0.0,NaN,0,98,0,98,0.0
2252,0,1.0,27,98,0.0,2400.0,0,98,0,98,0.0
3667,0,1.0,31,98,0.0,NaN,0,98,0,98,0.0
3937,1,1.0,39,98,19.0,NaN,0,98,0,98,0.0
4015,1,1.0,24,98,54.0,NaN,0,98,0,98,0.0
...,...,...,...,...,...,...,...,...,...,...,...
147974,1,1.0,55,98,0.0,NaN,0,98,0,98,0.0
148071,1,1.0,27,98,0.0,7840.0,0,98,0,98,0.0
148354,1,1.0,40,98,49.0,NaN,0,98,0,98,0.0
149290,1,1.0,51,98,0.0,NaN,0,98,0,98,0.0


In [70]:
df["age"].describe()


count    150000.000000
mean         52.295207
std          14.771866
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64

In [71]:
df["revolving_utilization_of_unsecured_lines"].describe()

count    150000.000000
mean          6.048438
std         249.755371
min           0.000000
25%           0.029867
50%           0.154181
75%           0.559046
max       50708.000000
Name: revolving_utilization_of_unsecured_lines, dtype: float64

In [72]:
df["debt_ratio"].describe()

count    150000.000000
mean        353.005076
std        2037.818523
min           0.000000
25%           0.175074
50%           0.366508
75%           0.868254
max      329664.000000
Name: debt_ratio, dtype: float64

In [73]:
df["number_of_times90_days_late"].describe()

count    150000.000000
mean          0.265973
std           4.169304
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          98.000000
Name: number_of_times90_days_late, dtype: float64

| Feature                     | Problem           | Why it’s wrong         |
| --------------------------- | ----------------- | ---------------------- |
| age                         | **min = 0**       | impossible             |
| revolving_utilization       | **max = 50,708**  | should be ≤ 1 normally |
| debt_ratio                  | **max = 329,664** | unrealistic            |
| number_of_times90_days_late | **max = 98**      | impossible             |


In [74]:
df[df["age"] == 0]

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
55001,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


In [75]:
df[df["revolving_utilization_of_unsecured_lines"] > 10].head()

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
1896,0,275.0,41,0,513.000000,NaN,3,0,1,0,0.0
4181,0,10151.0,35,0,0.341234,4260.0,10,0,0,0,0.0
4227,0,5649.0,63,0,0.287246,17625.0,6,0,1,0,1.0
4971,0,1761.0,64,0,0.082235,7587.0,2,0,1,0,0.0
5775,1,8328.0,39,0,21395.000000,NaN,9,0,2,1,0.0


In [76]:
df[df["debt_ratio"] > 100].head()

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
17,0,0.023967,37,0,408.0,NaN,6,0,0,0,0.0
20,0,0.001476,49,0,2940.0,NaN,5,0,2,0,0.0
23,0,0.019527,66,0,6049.0,NaN,12,0,4,0,0.0
27,0,0.146401,66,0,2735.0,NaN,16,0,1,0,0.0
31,0,0.003550,64,0,2471.0,NaN,11,0,2,0,1.0


In [77]:
df[df["number_of_times90_days_late"] > 20]

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
70,0,1.0,22,98,0.0,NaN,0,98,0,98,0.0
2252,0,1.0,27,98,0.0,2400.0,0,98,0,98,0.0
3667,0,1.0,31,98,0.0,NaN,0,98,0,98,0.0
3937,1,1.0,39,98,19.0,NaN,0,98,0,98,0.0
4015,1,1.0,24,98,54.0,NaN,0,98,0,98,0.0
...,...,...,...,...,...,...,...,...,...,...,...
147974,1,1.0,55,98,0.0,NaN,0,98,0,98,0.0
148071,1,1.0,27,98,0.0,7840.0,0,98,0,98,0.0
148354,1,1.0,40,98,49.0,NaN,0,98,0,98,0.0
149290,1,1.0,51,98,0.0,NaN,0,98,0,98,0.0


## Outlier handling

These are data corrections, not statistical estimates, so they don’t leak label information.

In [78]:
late_cols = [
    "number_of_time30_59_days_past_due_not_worse",
    "number_of_time60_89_days_past_due_not_worse",
    "number_of_times90_days_late",
]

df[late_cols] = df[late_cols].replace([96, 98], pd.NA)

In [79]:
df = df[df["age"] > 0]

In [80]:
df["age"].min()

np.int64(21)

In [81]:
df[late_cols].max()

number_of_time30_59_days_past_due_not_worse    13
number_of_time60_89_days_past_due_not_worse    11
number_of_times90_days_late                    17
dtype: object

In [82]:
df["revolving_utilization_of_unsecured_lines"].quantile(
    [0.90, 0.95, 0.99, 0.999]
)

0.900       0.981256
0.950       1.000000
0.990       1.092958
0.999    1571.012000
Name: revolving_utilization_of_unsecured_lines, dtype: float64

In [83]:
df["debt_ratio"].quantile(
    [0.90, 0.95, 0.99, 0.999]
)

0.900     1267.000
0.950     2449.000
0.990     4979.080
0.999    10613.148
Name: debt_ratio, dtype: float64

| Feature          | Observation | Meaning             |
| ---------------- | ----------- | ------------------- |
| utilization p99  | **1.09**    | normal credit usage |
| utilization p999 | **1571**    | corrupted values    |
| debt_ratio p99   | **4979**    | very skewed         |
| debt_ratio p999  | **10613**   | extreme outliers    |


### Clipping Outliers

| Method        | Problem               |
| ------------- | --------------------- |
| drop rows     | lose data             |
| log transform | harder interpretation |
| clipping      | stable + simple       |


In [84]:
util_cap = df["revolving_utilization_of_unsecured_lines"].quantile(0.99)
debt_cap = df["debt_ratio"].quantile(0.99)

df["revolving_utilization_of_unsecured_lines"] = df[
    "revolving_utilization_of_unsecured_lines"
].clip(upper=util_cap)

df["debt_ratio"] = df["debt_ratio"].clip(upper=debt_cap)

In [85]:
df["revolving_utilization_of_unsecured_lines"].max()

np.float64(1.0929580132799976)

In [86]:
df["debt_ratio"].max()

np.float64(4979.079999999958)

## Target feature relationship

In [87]:
df.groupby("number_of_times90_days_late")["serious_dlqin2yrs"].mean().reset_index().head(10)

,number_of_times90_days_late,serious_dlqin2yrs
0,0,0.046265
1,1,0.336639
2,2,0.499035
3,3,0.577211
4,4,0.670103
5,5,0.633588
6,6,0.600000
7,7,0.815789
8,8,0.714286
9,9,0.736842


In [88]:
df["utilization_bucket"] = pd.cut(
    df["revolving_utilization_of_unsecured_lines"],
    bins=[0, 0.2, 0.5, 0.8, 1.0, 2, 5],
)

In [89]:
df.groupby("utilization_bucket")["serious_dlqin2yrs"].mean()

utilization_bucket
(0.0, 0.2]    0.018735
(0.2, 0.5]    0.049624
(0.5, 0.8]    0.107768
(0.8, 1.0]    0.186190
(1.0, 2.0]    0.372478
Name: serious_dlqin2yrs, dtype: float64

| Utilization | Default rate |
| ----------- | ------------ |
| 0–20%       | **1.8%**     |
| 20–50%      | **4.9%**     |
| 50–80%      | **10.7%**    |
| 80–100%     | **18.6%**    |
| 100–200%    | **37.2%**    |


In [90]:
df["income_bucket"] = pd.qcut(df["monthly_income"], q=5)
df.groupby("income_bucket")["serious_dlqin2yrs"].mean()

income_bucket
(-0.001, 3000.0]       0.090775
(3000.0, 4544.0]       0.085954
(4544.0, 6300.0]       0.070128
(6300.0, 9083.0]       0.055306
(9083.0, 3008750.0]    0.045046
Name: serious_dlqin2yrs, dtype: float64

| Income    | Default rate |
| --------- | ------------ |
| ≤ 3000    | **9.0%**     |
| 3000–4544 | **8.6%**     |
| 4544–6300 | **7.0%**     |
| 6300–9083 | **5.5%**     |


### Linear Correlation

Measures how strongly two variables move together in a straight-line relationship.

Value ranges from -1 to 1:

- +1: when one increases, the other increases
- -1: when one increases, the other decreases
- 0: no linear relationship

In [91]:
late_cols = [
    "number_of_time30_59_days_past_due_not_worse",
    "number_of_time60_89_days_past_due_not_worse",
    "number_of_times90_days_late",
]

df[late_cols] = df[late_cols].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

In [92]:
df.corr(numeric_only=True)["serious_dlqin2yrs"].sort_values(ascending=False)

serious_dlqin2yrs                              1.000000
number_of_times90_days_late                    0.314535
revolving_utilization_of_unsecured_lines       0.280820
number_of_time30_59_days_past_due_not_worse    0.274553
number_of_time60_89_days_past_due_not_worse    0.268130
number_of_dependents                           0.046050
number_real_estate_loans_or_lines             -0.007037
debt_ratio                                    -0.017430
monthly_income                                -0.019746
number_of_open_credit_lines_and_loans         -0.029669
age                                           -0.115397
Name: serious_dlqin2yrs, dtype: float64

| Feature                                     | Correlation with Default | Interpretation                 |
| ------------------------------------------- | ------------------------ | ------------------------------ |
| number_of_times90_days_late                 | **0.31**                 | strongest signal               |
| revolving_utilization_of_unsecured_lines    | **0.28**                 | high credit usage = risk       |
| number_of_time30_59_days_past_due_not_worse | **0.27**                 | delinquency                    |
| number_of_time60_89_days_past_due_not_worse | **0.26**                 | delinquency                    |
| age                                         | **-0.11**                | younger borrowers default more |
